# GP06 — Filtering: Ενδεικτική λύση

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
df = pd.read_csv(Path('data/gp06/assets.csv'))
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326')


In [ ]:
active = gdf[gdf['status'].eq('active')]
priority = gdf[gdf['priority'].isin(['high', 'medium'])]
selected = gdf[gdf['status'].eq('active') & gdf['asset_type'].isin(['park', 'school']) & gdf['score'].between(75, 100)]
bbox = gdf.cx[23.55:23.60, 41.08:41.11]
print('active:', len(active), 'priority:', len(priority), 'selected:', len(selected), 'bbox:', len(bbox))


In [ ]:
needs_review = gdf[gdf['condition'].str.contains('review', case=False, na=False)]
not_inactive = gdf[~gdf['status'].eq('inactive')]
assert selected['status'].eq('active').all()
assert selected['asset_type'].isin(['park', 'school']).all()
assert selected['score'].between(75, 100).all()
assert gdf.crs.to_epsg() == 4326
assert gdf.geometry.notna().all()
print(selected[['asset_id', 'asset_type', 'score']])


In [ ]:
print('selected bounds:', selected.total_bounds)
assert selected.total_bounds[0] >= gdf.total_bounds[0]
assert selected.isna().sum().sum() == 0
selected.to_file('gp06_selected_assets.geojson', driver='GeoJSON')
print('exported:', len(selected), 'features')
